# A Feature Importance Score Is Not a Causal Effect

Article #29. Nine controlled synthetic experiments, nine original figures, and eighteen machine-readable evidence tables. This notebook uses fast independent smoke replications and isolates all outputs in a fresh temporary directory.


## Four different targets

1. **Predictive relevance:** does a feature improve future error under the specified evaluation design?
2. **Fixed-model reliance:** does changing the input change a fixed predictor's risk?
3. **Attribution:** which reference distribution and coalition game allocates a prediction among coordinates?
4. **Causal effect:** what happens to an outcome under a well-defined intervention in an identified causal system?

These answers do not become equal merely because all are called feature importance. The article derives the corresponding formulas.


In [ ]:
from pathlib import Path
import tempfile
import numpy as np
from IPython.display import Image, display
from analysis import Design, EXPERIMENTS, FIGURES, SEED, run
print('Seed:', SEED, '| studies:', ', '.join(EXPERIMENTS))


## Exact correlation and attribution identities

For an oracle f(x)=βx₁ and standardized bivariate normal features with correlation ρ, marginal permutation increases squared error in expectation by 2β²; conditional resampling of X₁ given X₂ increases it by 2β²(1−ρ²).

For f(x₁,x₂)=x₁ and centered Gaussian features, an interventional/marginal Shapley game yields (x₁,0), while a conditional/observational game yields (x₁−ρx₂/2, ρx₂/2). Both satisfy the coalition axioms for their *different* value functions; neither identifies a structural causal effect without more assumptions.


In [ ]:
from analysis import exact_two_feature_shapley
rho, x1, x2 = 0.8, 1.3, -0.4
for game in ('interventional', 'observational'):
    values = exact_two_feature_shapley(x1, x2, rho, game)
    assert abs(sum(values)-x1) < 1e-12
    print(game, values)


## Run all nine short, independent replicates

The canonical paper uses 18–24 independently seeded outer replications, depending on the experiment. This notebook uses three per study for speed. The generator, estimator and figure code are the *same functions*. The temporary destination ensures notebook executions do not overwrite evidence files.


In [ ]:
with tempfile.TemporaryDirectory() as directory:
    path = Path(directory)
    summaries = run(Design(root=path, quick=True))
    tables = sorted((path/'results').glob('*_quick.csv'))
    figures = sorted((path/'figures').glob('*_quick.png'))
    print('Evidence tables:', len(tables), '| original figures:', len(figures))
    assert len(tables) == 18 and len(figures) == 9
    for label, name in zip(EXPERIMENTS, FIGURES):
        print('\nStudy:', label, '| summary rows:', len(summaries[label]))
        display(Image(filename=str(path/'figures'/f'{name}_quick.png')))


## Reading the evidence responsibly

- All observations are synthetic, not field measurements.
- Error bars are 1.96 times the Monte Carlo standard error across independent outer draws; paired scenarios should not be analyzed as independent groups.
- Random-forest impurity credit and holdout permutation have different units and different selection mechanisms.
- Correlated predictors, interactions, post-outcome leakage, population shift, subgroup averaging, and winner selection can each change an importance story for a different reason.
- None of these simulations licenses a claim about the causal effect of an observed predictor. See the full manuscript for identification conditions and reporting guidance.
